# Phase 2 — Data Understanding (LendingClub)

**Goal:** profile the raw `accepted_2007_to_2018Q4.csv` so we know what we're modeling on, *before* touching any cleaning or features.

By the end of this notebook we will have:
1. A curated working dataset (≈30 columns, all available *at application time* — no leakage)
2. A precise definition of the target variable `default` derived from `loan_status`
3. The exact class imbalance ratio
4. A missingness profile per column
5. Univariate distributions of the headline features
6. An interim parquet saved to `data/interim/accepted_curated.parquet` for Phase 3 to pick up

**Why this matters:** in CRISP-DM, Data Understanding is where you discover the constraints that will shape Data Preparation. Skip it and you'll be debugging modeling bugs that are really data bugs.

## 0. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)
sns.set_theme(style='whitegrid')

PROJECT = Path(r'C:\Users\User\Desktop\Credit Risk Prediction System')
RAW = PROJECT / 'club loan data' / 'accepted_2007_to_2018q4.csv' / 'accepted_2007_to_2018Q4.csv'
INTERIM = PROJECT / 'data' / 'interim'
FIGS = PROJECT / 'outputs' / 'figures'
INTERIM.mkdir(parents=True, exist_ok=True)
FIGS.mkdir(parents=True, exist_ok=True)

assert RAW.exists(), f'Raw file not found: {RAW}'
print(f'Raw file size: {RAW.stat().st_size / 1e9:.2f} GB')

## 1. Curated column selection

The raw CSV has ~151 columns. Most are either:
- **Post-decision** (e.g. `total_pymnt`, `recoveries`, `last_pymnt_d`) — these encode the answer and would cause **target leakage**. We exclude them.
- **High-cardinality free text** (e.g. `emp_title`, `desc`, `url`) — low signal-to-noise unless you do NLP. Skip for now.
- **Near-duplicates of other columns** (e.g. `funded_amnt` ≈ `loan_amnt`). Pick one.

The 28 columns below are the standard application-time feature set used in published LendingClub default-modeling work. Each one is something a credit officer would see *before* approving the loan.

| Column | Why it matters |
|---|---|
| `loan_amnt`, `term`, `int_rate`, `installment` | Loan size and price |
| `grade`, `sub_grade` | LendingClub's own risk rating — strong signal but also a benchmark to beat |
| `emp_length`, `home_ownership`, `annual_inc`, `verification_status` | Income & employment stability |
| `purpose`, `addr_state`, `application_type` | Loan context |
| `dti`, `revol_util`, `revol_bal` | Borrower's existing debt load |
| `fico_range_low`, `fico_range_high` | Bureau credit score |
| `delinq_2yrs`, `pub_rec`, `pub_rec_bankruptcies`, `mort_acc`, `open_acc`, `total_acc` | Credit history depth & blemishes |
| `earliest_cr_line`, `issue_d` | Tenure of credit + when loan was made (needed for time-based split) |
| `loan_status` | **Target** |

In [ ]:
FEATURE_COLS = [
    'loan_amnt', 'term', 'int_rate', 'installment',
    'grade', 'sub_grade',
    'emp_length', 'home_ownership', 'annual_inc', 'verification_status',
    'purpose', 'addr_state', 'application_type',
    'dti', 'revol_util', 'revol_bal',
    'fico_range_low', 'fico_range_high',
    'delinq_2yrs', 'pub_rec', 'pub_rec_bankruptcies',
    'mort_acc', 'open_acc', 'total_acc',
    'earliest_cr_line', 'issue_d',
]
TARGET_COL = 'loan_status'
USECOLS = FEATURE_COLS + [TARGET_COL]
print(f'Loading {len(USECOLS)} columns of {USECOLS}')

## 2. Load

Reading only the curated columns drops memory from ~5 GB to ~400 MB. We also pass `low_memory=False` so pandas can infer dtypes from the whole column rather than chunk-by-chunk.

In [ ]:
%%time
df = pd.read_csv(RAW, usecols=USECOLS, low_memory=False)
print(f'Loaded shape: {df.shape}')
print(f'In-memory size: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')

## 3. Basic profile — shape, dtypes, head

In [ ]:
print(f'Rows:    {len(df):,}')
print(f'Columns: {df.shape[1]}')
print()
print('Dtypes:')
print(df.dtypes.value_counts())

In [ ]:
df.head()

## 4. Missingness profile

A column that is >50% NaN is rarely worth the imputation effort unless it's strongly predictive. We'll flag those for Phase 3.

In [ ]:
miss = df.isna().mean().sort_values(ascending=False) * 100
miss = miss[miss > 0]
print('Columns with any missing values (% NaN):')
print(miss.round(2).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(9, max(3, 0.3 * len(miss))))
miss.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('% missing')
ax.set_title('Missingness by column (curated features)')
fig.tight_layout()
fig.savefig(FIGS / 'phase2_missingness.png', dpi=120)
plt.show()

## 5. Target construction

`loan_status` has more than two values. Three groups matter:

| Group | Statuses | Treatment |
|---|---|---|
| **Default (1)** | `Charged Off`, `Default`, `Late (31-120 days)`, `Late (16-30 days)`, `Does not meet the credit policy. Status:Charged Off` | Map to 1 |
| **Repay (0)** | `Fully Paid`, `Does not meet the credit policy. Status:Fully Paid` | Map to 0 |
| **Censored** | `Current`, `In Grace Period`, `Issued` | **Drop** — outcome unknown |

Dropping the censored loans is the standard approach for binary classification when you have no time-to-event modeling in scope. The alternative is a survival model, which is overkill for this project.

**Why include `Late (16-30 days)` as default?** Because in production scoring, a borrower 16 days late is already a credit-risk event the bank must provision for under IFRS 9 Stage 2. Treating them as 'still good' would understate risk.

In [ ]:
print('Raw loan_status distribution:')
print(df['loan_status'].value_counts(dropna=False))

In [ ]:
DEFAULT_STATUSES = {
    'Charged Off', 'Default',
    'Late (31-120 days)', 'Late (16-30 days)',
    'Does not meet the credit policy. Status:Charged Off',
}
REPAY_STATUSES = {
    'Fully Paid',
    'Does not meet the credit policy. Status:Fully Paid',
}

def map_status(s):
    if s in DEFAULT_STATUSES:
        return 1
    if s in REPAY_STATUSES:
        return 0
    return np.nan  # censored — drop later

df['default'] = df['loan_status'].map(map_status)

n_before = len(df)
df = df.dropna(subset=['default']).copy()
df['default'] = df['default'].astype('int8')
n_after = len(df)
print(f'Dropped {n_before - n_after:,} censored rows ({(n_before - n_after)/n_before*100:.1f}%). Remaining: {n_after:,}')

## 6. Class imbalance

In [ ]:
counts = df['default'].value_counts()
pct = df['default'].value_counts(normalize=True) * 100
print('Class counts:')
print(counts.to_string())
print()
print(f'Default rate: {pct[1]:.2f}%')
print(f'Imbalance ratio (repay : default) ≈ {counts[0] / counts[1]:.1f} : 1')

In [ ]:
# Default rate by grade and by year
df['issue_year'] = pd.to_datetime(df['issue_d'], format='%b-%Y', errors='coerce').dt.year

by_grade = df.groupby('grade')['default'].agg(['mean', 'count']).rename(columns={'mean': 'default_rate'})
by_grade['default_rate'] = (by_grade['default_rate'] * 100).round(2)
print('Default rate by LC grade:')
print(by_grade.to_string())

by_year = df.groupby('issue_year')['default'].agg(['mean', 'count']).rename(columns={'mean': 'default_rate'})
by_year['default_rate'] = (by_year['default_rate'] * 100).round(2)
print()
print('Default rate by issue year:')
print(by_year.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
by_grade['default_rate'].plot(kind='bar', ax=axes[0], color='tomato')
axes[0].set_title('Default rate by LC grade')
axes[0].set_ylabel('% default')
by_year['default_rate'].plot(kind='line', marker='o', ax=axes[1], color='steelblue')
axes[1].set_title('Default rate by issue year')
axes[1].set_ylabel('% default')
fig.tight_layout()
fig.savefig(FIGS / 'phase2_default_by_grade_year.png', dpi=120)
plt.show()

## 7. Univariate profile of headline features

In [ ]:
num_cols = ['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'fico_range_low', 'revol_util']
df[num_cols].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).round(2)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flat, num_cols):
    # Clip to 1st-99th percentile so plots aren't dominated by extreme outliers
    lo, hi = df[col].quantile([0.01, 0.99])
    s = df[col].clip(lo, hi)
    ax.hist(s.dropna(), bins=50, color='steelblue', alpha=0.7)
    ax.set_title(col)
fig.suptitle('Univariate distributions (clipped 1–99th pct)')
fig.tight_layout()
fig.savefig(FIGS / 'phase2_univariate.png', dpi=120)
plt.show()

In [ ]:
# Default rate by FICO bucket — sanity-check that higher FICO → lower default
fico_bins = pd.cut(df['fico_range_low'], bins=[600, 650, 680, 700, 720, 750, 850])
fico_default = df.groupby(fico_bins, observed=True)['default'].agg(['mean', 'count'])
fico_default['mean'] = (fico_default['mean'] * 100).round(2)
print('Default rate by FICO bucket (sanity check — should decrease with FICO):')
print(fico_default.to_string())

## 8. Save interim parquet for Phase 3

Parquet is ~10× smaller than CSV and ~100× faster to reload, with dtypes preserved. From here on, Phases 3–6 read from this file.

In [ ]:
out_path = INTERIM / 'accepted_curated.parquet'
df.to_parquet(out_path, index=False)
print(f'Saved {len(df):,} rows × {df.shape[1]} cols → {out_path}')
print(f'File size: {out_path.stat().st_size / 1e6:.1f} MB')

## 9. Phase 2 findings — what we now know

Fill these in *after* running the cells above (the numbers in brackets are placeholders):

1. **Working dataset size**: ~[X.X]M rows × 28 columns after dropping censored loans.
2. **Default rate**: ~[X]%, imbalance ratio ≈ [X]:1 → Phase 3 must address with class weights / SMOTE.
3. **Heaviest missingness**: `[col]` ([X]%), `[col]` ([X]%) — decide drop vs impute in Phase 3.
4. **Sanity check** — default rate falls monotonically from grade G to A (✓ / ✗) and from low FICO to high FICO (✓ / ✗). If either fails, something is wrong with the data.
5. **Time pattern** — default rate spiked in [year(s)], reflecting [reason]. Consider a time-based train/test split, not random.

---

### Check-for-understanding (answer before Phase 3)

1. Why did we **drop** rows where `loan_status == 'Current'` rather than label them 0 (repay)?
2. Why is **`total_pymnt` excluded** from `FEATURE_COLS`? What would happen to model performance if you accidentally included it?
3. Looking at the default rate by grade table, what does it tell you about whether LendingClub's own grading system is informative?
4. Given the default rate you observed, what's the **accuracy of a 'predict everyone repays' baseline**? Compare to the headline number to confirm why accuracy alone is a bad metric (callback to Phase 1).
5. Would you use a **random** train/test split or a **time-based** split for this dataset, and why?